In [1]:
import pandas as pd

train = pd.read_csv("data/train.csv")
product = pd.read_csv("data/product_master.csv")
sub = pd.read_csv("data/submission.csv")

In [2]:
from feature import fill_missing_values_no_drop
from feature import expand_grid_train_only
from feature import add_advanced_lifecycle_features

# 1. Temizlik
train, product = fill_missing_values_no_drop(train, product)

# 2. Genişletme (Zaman serisi onarımı)
train = expand_grid_train_only(train)

# 3. Özellik Türetme (Logaritmik ve Urgency dahil)
train = add_advanced_lifecycle_features(train, product)

🧹 1. ADIM: Product Temizliği...
🚀 2. ADIM: Train Grid Genişletme (Giriş: 278901)...
🧬 3. ADIM: Lifecycle Özellikleri & Gürültü Temizliği...
✂️  Üretim öncesi 134469 adet 'yapay' satır silindi.


## 🔄 Çapraz Doğrulama ile Model Değerlendirme

In [3]:
from utils import evaluate_model_cv
from model import train_model

# Çapraz doğrulama ile model performansını değerlendir
cv_results = evaluate_model_cv(
    train_df=train,
    product_df=product,
    model_func=train_model,
    n_splits=3,
    test_months=12
)

print("\n📊 Detaylı CV Sonuçları:")
print(f"Fold rWMAPE'ler: {[f'{x:.4f}' for x in cv_results['fold_rwmapes']]}")
print(f"Fold Skorlar: {[f'{x:.4f}' for x in cv_results['fold_competition_scores']]}")


🔄 Çapraz Doğrulama Başlıyor (3 fold)

📊 Fold 1/1
   Train: 2022-01 - 2023-10
   Val:   2023-11 - 2024-10

🚀 Model Eğitimi Başlıyor
🔧 Özellikler hazırlanıyor...
✅ Özellikler hazır!
📊 Eğitim verisi: 320624 satır, 23 özellik
🤖 LightGBM modeli eğitiliyor...
✅ Model eğitimi tamamlandı!

📅 12 ay için tahmin yapılıyor (Batch mode)...
🚀 Batch prediction yapılıyor: 212496 satır...
✅ Tahminler tamamlandı: 212496 satır

🔝 En Önemli 15 Özellik:
rolling_mean_3                 :     1171
lag_1                          :      881
lag_2                          :      879
rolling_std_3                  :      620
lag_3                          :      347
month                          :      289
rolling_mean_6                 :      253
rolling_std_6                  :      240
category_encoded               :      221
month_cos                      :      182
lag_6                          :      180
rolling_std_12                 :      131
month_sin                      :      116
lag_12          

## 🎯 Final Model Eğitimi ve Submission Oluşturma

In [4]:
from model import train_and_predict_full
from utils import prepare_submission_format

# Tüm veri ile final model eğit
print("\n" + "="*60)
print("🏆 FINAL MODEL EĞİTİMİ")
print("="*60)

final_predictions = train_and_predict_full(
    train_df=train,
    product_df=product,
    submission_template_path='data/submission.csv'
)

print(f"\n✅ Tahminler hazır: {len(final_predictions)} satır")
print(f"📊 Tahmin özeti:")
print(final_predictions['quantity'].describe())


🏆 FINAL MODEL EĞİTİMİ

🎯 Forecast Dönemi: 2024-11 - 2025-10

🚀 Model Eğitimi Başlıyor
🔧 Özellikler hazırlanıyor...
✅ Özellikler hazır!
📊 Eğitim verisi: 550563 satır, 23 özellik
🤖 LightGBM modeli eğitiliyor...
✅ Model eğitimi tamamlandı!

📅 12 ay için tahmin yapılıyor (Batch mode)...
🚀 Batch prediction yapılıyor: 241776 satır...
✅ Tahminler tamamlandı: 241776 satır

🔝 En Önemli 15 Özellik:
rolling_mean_3                 :     1168
lag_1                          :      923
lag_2                          :      826
rolling_std_3                  :      569
lag_3                          :      306
category_encoded               :      267
rolling_mean_6                 :      255
month                          :      254
rolling_std_6                  :      205
market_encoded                 :      200
lag_6                          :      195
month_cos                      :      184
month_sin                      :      128
rolling_mean_12                :      117
brand_encoded      

In [5]:
# Submission formatına dönüştür
submission_df = prepare_submission_format(
    predictions_df=final_predictions,
    submission_template_path='data/submission.csv'
)

# Kaydet
submission_df.to_csv('data/my_submission.csv', index=False)

print("\n" + "="*60)
print("💾 SUBMISSION DOSYASI OLUŞTURULDU")
print("="*60)
print(f"📁 Dosya: data/my_submission.csv")
print(f"📊 Satır sayısı: {len(submission_df)}")
print(f"\n📈 Submission özeti:")
print(submission_df['quantity'].describe())
print(f"\n🔍 İlk 10 satır:")
print(submission_df.head(10))


💾 SUBMISSION DOSYASI OLUŞTURULDU
📁 Dosya: data/my_submission.csv
📊 Satır sayısı: 249996

📈 Submission özeti:
count    249996.000000
mean         61.415174
std         299.183328
min           0.000000
25%           0.082425
50%           0.938125
75%          19.202395
max       16023.342746
Name: quantity, dtype: float64

🔍 İlk 10 satır:
   ID       unique_code       date    quantity
0   0  MKT_001-PRD_0010 2024-11-01  810.959024
1   1  MKT_001-PRD_0010 2024-12-01  810.959024
2   2  MKT_001-PRD_0010 2025-01-01  764.076871
3   3  MKT_001-PRD_0010 2025-02-01  764.076871
4   4  MKT_001-PRD_0010 2025-03-01  795.700654
5   5  MKT_001-PRD_0010 2025-04-01  795.700654
6   6  MKT_001-PRD_0010 2025-05-01  795.700654
7   7  MKT_001-PRD_0010 2025-06-01  801.269780
8   8  MKT_001-PRD_0010 2025-07-01  801.269780
9   9  MKT_001-PRD_0010 2025-08-01  801.269780


## ✅ Submission Doğrulama

In [6]:
# Submission formatını kontrol et
original_sub = pd.read_csv('data/submission.csv')
my_sub = pd.read_csv('data/my_submission.csv')

print("🔍 Submission Doğrulama:")
print(f"✓ Satır sayısı eşleşiyor mu? {len(original_sub) == len(my_sub)}")
print(f"✓ Sütunlar doğru mu? {list(original_sub.columns) == list(my_sub.columns)}")
print(f"✓ Negatif değer var mı? {(my_sub['quantity'] < 0).sum() == 0}")
print(f"✓ NaN değer var mı? {my_sub['quantity'].isna().sum() == 0}")
print(f"\n📊 Quantity dağılımı:")
print(f"   Min: {my_sub['quantity'].min():.2f}")
print(f"   Max: {my_sub['quantity'].max():.2f}")
print(f"   Mean: {my_sub['quantity'].mean():.2f}")
print(f"   Median: {my_sub['quantity'].median():.2f}")
print(f"   Sıfır tahmin oranı: {(my_sub['quantity'] == 0).sum() / len(my_sub) * 100:.1f}%")

🔍 Submission Doğrulama:
✓ Satır sayısı eşleşiyor mu? True
✓ Sütunlar doğru mu? True
✓ Negatif değer var mı? True
✓ NaN değer var mı? True

📊 Quantity dağılımı:
   Min: 0.00
   Max: 16023.34
   Mean: 61.42
   Median: 0.94
   Sıfır tahmin oranı: 3.7%
